In [1]:
from google.colab import files

uploaded = files.upload()

Saving train.csv to train.csv


In [2]:
"""LLM 작성 모델 분류 프로젝트: 데이터 검사 및 Train/Val/Test 분할 (Google Colab용).

사용 방법:
1. Google Colab에 원본 train.csv와 이 파일을 업로드합니다.
2. 코드 셀에서 %run split_data_final.py 를 실행합니다.
3. split_data/ 폴더에 저장된 CSV 파일 3개가 자동 다운로드됩니다.
"""

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

# ==========================================
# 1. 기본 설정
# ==========================================

# 원본 데이터셋 파일 이름
INPUT_FILE = "train.csv"

# 분할된 파일을 저장할 폴더 (원본 파일 덮어쓰기 방지)
OUTPUT_DIR = Path("split_data")

# 실행할 때마다 동일한 분할 결과가 나오도록 시드 고정
SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ==========================================
# 2. 원본 데이터 불러오기
# ==========================================

df = pd.read_csv(INPUT_FILE)

# 필수 컬럼 존재 여부 확인
required = {"id", "prompt_id", "text", "model"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"필수 컬럼 누락: {sorted(missing)}")

print("=== 데이터셋 기본 정보 ===")
print(f"전체 데이터 수: {len(df)}")
print(f"프롬프트 수: {df['prompt_id'].nunique()}")
print(f"LLM 모델 수: {df['model'].nunique()}")
print(f"컬럼 목록: {list(df.columns)}")


# ==========================================
# 3. 데이터 품질 검사
# ==========================================

print("\n=== 결측치 검사 ===")
print(df.isnull().sum())

print("\n=== 중복 검사 ===")
print(f"완전 중복 행: {df.duplicated().sum()}")
print(f"중복 ID: {df['id'].duplicated().sum()}")
print(f"중복 텍스트: {df['text'].duplicated().sum()}")

print("\n=== 원본 모델별 데이터 수 ===")
print(df["model"].value_counts().sort_index())

# 결측치 또는 중복 ID가 발견되면 임의 삭제 없이 실행 중단
if df[list(required)].isnull().any().any():
    raise ValueError("필수 컬럼에 결측치가 있습니다. 확인 후 다시 실행하세요.")
if not df["id"].is_unique:
    raise ValueError("중복 ID가 존재합니다. 확인 후 다시 실행하세요.")


# ==========================================
# 4. prompt_id 기준 데이터 분할
# ==========================================

# 같은 프롬프트에 대한 모든 LLM 응답은 동일한 데이터셋에 배치
prompt_ids = np.sort(df["prompt_id"].unique())

# 프롬프트 기준 Train 70% / 임시 데이터 30%
train_ids, temp_ids = train_test_split(
    prompt_ids,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
)

# 임시 데이터 30%를 절반으로 나눠 Val 15% / Test 15%
val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=SEED,
    shuffle=True,
)

# 각 프롬프트에 속한 모든 행 추출
train_df = df[df["prompt_id"].isin(train_ids)].copy()
val_df = df[df["prompt_id"].isin(val_ids)].copy()

=== 데이터셋 기본 정보 ===
전체 데이터 수: 2699
프롬프트 수: 150
LLM 모델 수: 18
컬럼 목록: ['id', 'prompt_id', 'text', 'model']

=== 결측치 검사 ===
id           0
prompt_id    0
text         0
model        0
dtype: int64

=== 중복 검사 ===
완전 중복 행: 0
중복 ID: 0
중복 텍스트: 0

=== 원본 모델별 데이터 수 ===
model
claude-fable-5            149
claude-opus-4-6           150
claude-opus-4-8           150
claude-opus-5             150
claude-sonnet-5           150
gemini-3.1-pro-preview    150
gemini-3.6-flash          150
gemini-3.7-flash          150
gemini-3.8-flash          150
glm-5.3                   150
glm-5.3-flash             150
gpt-5.5                   150
gpt-5.6-luna              150
gpt-5.6-sol               150
gpt-5.6-terra             150
gpt-6-astra               150
kimi-k3                   150
qwen3.8-max               150
Name: count, dtype: int64
